In [29]:
%%capture
%pip install python-dotenv
import dotenv
%load_ext dotenv
%dotenv

In [30]:
%%capture
!pip install crewai crewai-tools langchain-openai duckduckgo-search langchain-community ddgs

# Geração de Roteiro e Thumbnails para Vídeos

## Descrição

O objetivo deste desafio é criar um sistema automatizado que gere roteiros para vídeos no segmento de cinema. Com base no roteiro, o sistema também deve produzir três opções de thumbnails inspiradas no conteúdo do vídeo e escolher uma dessas thumbnails.

---

## Estrutura do Projeto

### Agentes

#### Roteirista de Vídeo

Responsável por pesquisar e elaborar um roteiro detalhado para um vídeo completo no YouTube.
Possui acesso a ferramentas de pesquisa para enriquecer seu conhecimento.
Especialista em criação de conteúdo e storytelling para cinema.

#### Criador de Thumbnail

Utiliza o roteiro gerado para produzir três opções de thumbnails.
As thumbnails devem ser inspiradas no conteúdo do vídeo, destacando elementos visuais atrativos.
Designer gráfico com experiência em thumbnails chamativas para YouTube.

#### Revisor

Revisar o texto do roteiro
Escrever a versão contendo roteiro + thumbnails

---

## Execução

O sistema pode ser acionado fornecendo um tema de vídeo, como por exemplo:

```python
inputs={'query': 'Melhores filmes de 2025'}
```

---

In [31]:
import os

from crewai import Agent, Task, Crew, Process
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from crewai_tools import DallETool
from langchain_openai import ChatOpenAI

from IPython.display import Markdown

In [32]:

llm = ChatOpenAI(model="gpt-4")

In [38]:
from crewai.tools import tool

@tool
def search_duckduckgo(query: str) -> str:
    """Pesquisa informações na web usando DuckDuckGo. Aceita um argumento de texto como consulta."""
    from langchain_community.tools import DuckDuckGoSearchRun
    
    if not query or not isinstance(query, str):
        raise ValueError("O argumento 'query' deve ser uma string não vazia.")
    
    searcher = DuckDuckGoSearchRun()
    return searcher.invoke(query)

In [39]:
script_writer = Agent(
    name="Roteirista",
    role="Roteirista",
    goal="""Escrever roteiro sobre {query}. Esse roteiro deve estar relacionado ao mundo de cinema.
    Seu roteiro deve ser conciso e conter uma estrutura dinâmica com no mínimo Introdução, Tema Principal e Conclusão.
    """,
    backstory="Um redator experiente que já escreveu diversos roteiros e artigos sobre o mundo do cinema.",
    tools=[search_duckduckgo],
    max_iter=3,
    allow_delegation=False,
    memory=True,
    verbose=True
)

In [40]:
script_task = Task(
    description="Pesquisar sobre o tema e criar um roteiro convincente para",
    expected_output="""Escrever um roteiro sobre {query}. Utilizando uma estrutura concisa de Introdução, Tema Principal e Conclusão.
    Sua saída deve estar em formato Markdown para publicação do roteiro.
    """,
    agent=script_writer,
)

In [41]:
thumb_maker = Agent(
    name="Thumb Maker",
    role="Thumb Maker",
    goal="""Você deve montar 3 prompts diferentes para gerar diferentes imagens que serão escolhidas para serem thumbnail de um vídeo gravado com base no roteiro.
    A imagens devem levar em conta tanto o roteiro como o tema {query}. Cada prompt para gerar as imagens deve ser único e relacionado
    a diferentes representações possíveis do tema.
    """,
    backstory="Designer gráfico com experiência na criação de thumbnails chamativas para YouTube.",
    tools=[DallETool()],
    max_iter=3,
    allow_delegation=False,
    memory=True,
    verbose=True
)

In [42]:
thumbnail_task = Task(
    description='Com base no roteiro do vídeo, gerar 3 opções de thumbnail inspiradas no conteúdo e retorná-las como imagens.',
    expected_output="""Na saída devem existir 3 imagens com diferentes expressões artísticas que estejam relacionadas ao roteiro e ao tema.
    Cada uma das imagens devem estar em 1920x1080 e possuir um texto chamativo na imagem.
    """,
    agent=thumb_maker,
    context=[script_task],
)

In [43]:
reviwer = Agent(
    name="Revisor",
    role="Revisor",
    goal="""Você deve revisar o texto de roteiro e caso necessário solicitar uma melhora do texto ao Roteirista. Além disso você deve juntar o roteiro final com as opçõe de
    thumbnails geradas pelo Thumb Maker. 
    """,
    backstory="Você é um revisor de roteiro com amplo conhecimento em escrita e domínio em temas de design e marketing para gerar vídeos relevantes",
    max_iter=3,
    allow_delegation=True,
    memory=True,
    verbose=True
)

In [44]:
review_task = Task(
    description='Gerar um roteiro final juntando tanto o roteiro quanto as opções de thumbnails disponibilizadas',
    expected_output="""Sua saída deve estar em formato markdown e juntar o roteiro com opções de thumbnail.
    As imagens devem ser encontradas de forma sequencial após o roteiro no final do texto.
    """,
    agent=reviwer,
    context=[script_task, thumbnail_task],
)

In [45]:
crew = Crew(
    agents=[script_writer, thumb_maker, reviwer],
    tasks=[script_task, thumbnail_task, review_task],
    verbose=True,
    process=Process.hierarchical,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
)

In [46]:
result = crew.kickoff(inputs={"query": "Melhores filmes de 2024"})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c2b5c1ba-44ed-49fc-ab5e-8faf72863d71                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Pesquisar sobre o tema e criar um roteiro convincente para                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/luiz-augusto/Documents/StudioAgent-Cinema/.venv/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


/home/luiz-augusto/Documents/StudioAgent-Cinema/.venv/lib/python3.12/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Thought: Preciso buscar informações detalhadas sobre os melhores filmes de 2024 para poder fornecer ao         │
│  roteirista todo o conteúdo necessário e ele poder criar um roteiro convincente.                                │
│                                                                                                                 │
│  Using Tool: search_duckduckgo                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

🚀 Crew: crew
├── 📋 Task: 285f1158-ded4-4d13-b4da-994c95dfea5b
│   Assigned to: Crew Manager
│   Status: ✅ Completed
│   ├── 🔧 Used search_duckduckgo (1)
│   ├── 🔧 Used search_duckduckgo (2)
│   ├── 🔧 Using Delegate work to coworker (1)
│   └── 🔧 Used search_duckduckgo (3)
├── 📋 Task: 721d8305-f4b8-40c8-b471-1eb98fb8ecf0
│   Assigned to: Crew Manager
│   Status: ✅ Completed
│   ├── 🔧 Using Delegate work to coworker (2)
│   ├── 🔧 Used Dall-E Tool (1)
│   ├── 🔧 Used Dall-E Tool (2)
│   ├── 🔧 Used Dall-E Tool (3)
│   ├── 🔧 Used Dall-E Tool (4)
│   ├── 🔧 Using Delegate work to coworker (3)
│   ├── 🔧 Used Dall-E Tool (5)
│   ├── 🔧 Used Dall-E Tool (6)
│   └── 🔧 Used Dall-E Tool (7)
└── 📋 Task: 6cf58900-229c-4db8-96d7-b6f290c4eacb
    Status: Executing Task...
    └── 🔧 Using Delegate work to coworker (4)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Review of the Best Movies of 2024                                                                            │
│                                                                                                                 │
│  ## Joy                                                                                                         │
│                                                                                                                 │
│  **Storyline:**                                                                                                 │
│  "Joy" is a heartwarming drama that follows the life of a young woman overcoming personal struggles to find     │
│  her true passion and happiness. Set against the vibrant backdrop of a bustling city, the film explores themes  │
│  of resilience, self-discovery, and the power of hope. As Joy navigates challenges in both her professional     │
│  and personal life, she learns valuable lessons about trust, love, and the importance of pursuing one’s dreams  │
│  no matter the obstacles.                                                                                       │
│                                                                                                                 │
│  **Thumbnail Description:**                                                                                     │
│  A vibrant, uplifting image featuring the protagonist Joy standing confidently amidst a bustling city skyline   │
│  at sunset. Warm hues of orange and pink cast a hopeful glow, emphasizing her journey towards                   │
│  self-empowerment. The text overlay reads, “Joy – A Story of Hope and Resilience.”                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Ainda Estou Aqui                                                                                            │
│                                                                                                                 │
│  **Storyline:**                                                                                                 │
│  "Ainda Estou Aqui" is a compelling narrative centered around a middle-aged man who grapples with identity and  │
│  belonging after a life-changing event. The film delves deep into themes of memory, reconciliation, and         │
│  personal growth as the protagonist reconnects with his roots and those he left behind. Set in a picturesque    │
│  coastal town, the story intertwines moments of introspection with striking visuals that highlight the beauty   │
│  of rediscovery and second chances.                                                                             │
│                                                                                                                 │
│  **Thumbnail Description:**                                                                                     │
│  A contemplative, serene image displaying the main char

🚀 Crew: crew
├── 📋 Task: 285f1158-ded4-4d13-b4da-994c95dfea5b
│   Assigned to: Crew Manager
│   Status: ✅ Completed
│   ├── 🔧 Used search_duckduckgo (1)
│   ├── 🔧 Used search_duckduckgo (2)
│   ├── 🔧 Using Delegate work to coworker (1)
│   └── 🔧 Used search_duckduckgo (3)
├── 📋 Task: 721d8305-f4b8-40c8-b471-1eb98fb8ecf0
│   Assigned to: Crew Manager
│   Status: ✅ Completed
│   ├── 🔧 Using Delegate work to coworker (2)
│   ├── 🔧 Used Dall-E Tool (1)
│   ├── 🔧 Used Dall-E Tool (2)
│   ├── 🔧 Used Dall-E Tool (3)
│   ├── 🔧 Used Dall-E Tool (4)
│   ├── 🔧 Using Delegate work to coworker (3)
│   ├── 🔧 Used Dall-E Tool (5)
│   ├── 🔧 Used Dall-E Tool (6)
│   └── 🔧 Used Dall-E Tool (7)
└── 📋 Task: 6cf58900-229c-4db8-96d7-b6f290c4eacb
    Assigned to: Crew Manager
    Status: ✅ Completed
    └── 🔧 Using Delegate work to coworker (4)

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6cf58900-229c-4db8-96d7-b6f290c4eacb                                                                     │
│  Agent: Crew Manager                                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been

In [47]:
Markdown(result.raw)

```markdown
# Review of the Best Movies of 2024

## Joy

**Storyline:**  
"Joy" is a heartwarming drama that follows the life of a young woman overcoming personal struggles to find her true passion and happiness. Set against the vibrant backdrop of a bustling city, the film explores themes of resilience, self-discovery, and the power of hope. As Joy navigates challenges in both her professional and personal life, she learns valuable lessons about trust, love, and the importance of pursuing one’s dreams no matter the obstacles.

**Thumbnail Description:**  
A vibrant, uplifting image featuring the protagonist Joy standing confidently amidst a bustling city skyline at sunset. Warm hues of orange and pink cast a hopeful glow, emphasizing her journey towards self-empowerment. The text overlay reads, “Joy – A Story of Hope and Resilience.”

---

## Ainda Estou Aqui

**Storyline:**  
"Ainda Estou Aqui" is a compelling narrative centered around a middle-aged man who grapples with identity and belonging after a life-changing event. The film delves deep into themes of memory, reconciliation, and personal growth as the protagonist reconnects with his roots and those he left behind. Set in a picturesque coastal town, the story intertwines moments of introspection with striking visuals that highlight the beauty of rediscovery and second chances.

**Thumbnail Description:**  
A contemplative, serene image displaying the main character gazing out towards the ocean at dusk, with soft blues and purples dominating the palette. The mood reflects introspection and peace. The text overlay says, “Ainda Estou Aqui – Embracing the Past, Finding the Future.”

---

## General Theme Thumbnail

**Thumbnail Description:**  
An elegant collage featuring iconic elements from both "Joy" and "Ainda Estou Aqui," blending cityscapes with coastal scenes in a harmonious color scheme. The central text boldly states: “Best Films of 2024 – A Cinematic Journey,” inviting viewers to explore the year's standout cinematic experiences.
```